# 14 · Structured Streaming — Preço Médio Ponderado de NFe (CSV → Parquet)

🎯 **Objetivo:** simular a chegada de itens de **notas fiscais eletrônicas** (NFe) em lotes de CSV a cada 5 minutos e manter uma tabela Gold com o **preço médio praticado por produto e lojista**, atualizada a cada lote — sem nunca reescrever o histórico inteiro, e sem deixar um único lote (bom ou ruim) definir o preço sozinho.

**Teoria:** docs/09-spark-streaming.md · **Pré-requisitos:** notebooks 12 e 13 (janelas, watermarks, file source, agregações estatísticas)

⚠️ **Este notebook substitui a abordagem de janela fixa por um acumulador com estado.** A primeira versão deste laboratório usava `window(1 hour, 15 minutes)` — cada evento contribuía para até 4 janelas sobrepostas, e cada janela era recalculada do zero a partir dos dados brutos. Isso funciona bem para "quanto vendemos entre 10h e 11h", mas não é a pergunta certa aqui: queremos "qual é o preço **praticado** por este lojista, considerando o dia inteiro até agora?" — uma pergunta que precisa de **memória entre lotes**, não de uma janela que fecha e é esquecida.

A solução: `applyInPandasWithState` — um acumulador **por produto e lojista** que carrega para frente, lote a lote, dois números só (valor acumulado e quantidade acumulada), aplicando um **decaimento exponencial** a cada atualização. Isso resolve os dois lados do problema pedido:

1. **Não sofrer efeito de outliers/inliers:** um lote de 5 minutos nunca é a "verdade" sozinho — ele só *ajusta* um acumulado que já carrega a história do dia.
2. **Refletir a prática do lojista ao longo do tempo:** o decaimento garante que a história muito antiga vá perdendo peso — se o lojista genuinamente mudar de preço no meio do dia, o acumulado eventualmente acompanha, em vez de ficar preso para sempre à média da manhã.

---
### 🔤 O que você vai praticar

1. **Watermark de 1 hora** para tolerar notas que chegam atrasadas — bem mais generoso que a janela do próprio negócio (lotes a cada 5 minutos)
2. **`applyInPandasWithState`** — estado arbitrário por chave (`gtin`, `cnpj_lojista`), carregado lote a lote, sem reprocessar o dia inteiro a cada atualização
3. **Média ponderada com decaimento exponencial** — por que "considerar a janela anterior" é uma fórmula recursiva, não um recálculo do zero
4. **`GroupStateTimeout.EventTimeTimeout`** — encerrar e liberar o estado de um produto/lojista que fica muito tempo sem vender (reaproveitando o mesmo watermark de 1h)
5. **`foreachBatch` + Parquet como log histórico** — já que o sink `parquet` só aceita `append`, tratamos a tabela Gold como uma sequência de instantâneos, e lemos "o preço atual" pegando o mais recente por chave

Vamos simular o fluxo!

In [ ]:
import os
import sys

from pyspark.sql import SparkSession

# Fixa o worker Python na mesma versão do driver ANTES de criar a SparkSession — evita
# erro de versão quando o `python3` do sistema é mais novo que o do venv (comum com Homebrew).
# Precisa ser variável de ambiente: a config `spark.pyspark.python` sozinha não é suficiente.
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = (
    SparkSession.builder.appName("app-01")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.executor.memory", "2g")
    .config("spark.executor.cores", "2")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")  # silencia o ruído de INFO/WARN de cada micro-batch

spark

## O domínio: itens de nota fiscal eletrônica

Cada linha do CSV é um **item** de uma NFe — o mesmo `id` de nota se repete em todas as linhas que pertencem a ela, exatamente como o XML da NFe real repete a tag `<det>` uma vez por produto vendido. Na vida real, o sistema do lojista transmite um **lote novo a cada 5 minutos** — é esse ritmo que vamos simular.

| Campo | O que é |
|---|---|
| `id` | Identificador da nota — **repete** entre os itens da mesma nota |
| `timestamp` | Instante da venda (nosso event time) |
| `gtin` | Código de barras do produto (identifica o "o quê") |
| `cnpj_lojista` | CNPJ do estabelecimento (identifica o "quem vendeu") |
| `preco_unitario` | Preço cobrado por unidade nesta venda |
| `unidade_medida` | Unidade de venda (`UN`, `KG`, etc.) |
| `quantidade_comprada` | Quantas unidades desse produto vieram nesta nota |
| `id_municipio` / `nome_municipio` | Onde fica o lojista (dado descritivo — não entra na agregação) |

📌 `gtin` + `cnpj_lojista` continuam sendo a chave de negócio: "preço médio **deste produto, neste lojista**" — só que agora essa chave carrega um **estado que atravessa o dia inteiro**, em vez de viver e morrer dentro de uma janela de tempo fixa.

In [ ]:
import csv
import os
import shutil
import time
from datetime import datetime, timedelta

import numpy as np
from pyspark.sql.types import DoubleType, StringType, StructField, StructType, TimestampType

# Toda a simulação vive sob esta pasta — limpa a cada execução do notebook,
# para que o resultado seja sempre reproduzível.
BASE = "../data/streaming/nb14"
CAMINHO_GOLD = "../data/gold/nb14_preco_medio_produto_lojista_hora"
shutil.rmtree(BASE, ignore_errors=True)
shutil.rmtree(CAMINHO_GOLD, ignore_errors=True)

CAMINHO_LANDING = f"{BASE}/landing_nfe"
CAMINHO_CHECKPOINT_GOLD = f"{BASE}/_checkpoint_gold"
os.makedirs(CAMINHO_LANDING, exist_ok=True)

COLUNAS_NFE = [
    "id", "timestamp", "gtin", "cnpj_lojista", "preco_unitario",
    "unidade_medida", "quantidade_comprada", "id_municipio", "nome_municipio",
]

# Catálogo de produtos e lojistas — só existe no lado Python (não faz parte do CSV),
# é o que usamos para gerar dados sintéticos plausíveis e para nomear as coisas nos comentários.
PRODUTOS = {
    "7891000100103": {"nome": "Arroz Tipo 1 5kg", "unidade": "UN", "preco_base": 25.00},
    "7891000200208": {"nome": "Feijão Carioca 1kg", "unidade": "UN", "preco_base": 7.50},
    "7891000300303": {"nome": "Óleo de Soja 900ml", "unidade": "UN", "preco_base": 8.20},
}
GTIN_ARROZ = "7891000100103"
GTIN_FEIJAO = "7891000200208"

LOJISTAS = {
    "11222333000181": {"nome": "Mercado Bom Preço", "id_municipio": "3550308", "nome_municipio": "São Paulo"},
    "44555666000162": {"nome": "Supermercado Estrela", "id_municipio": "3304557", "nome_municipio": "Rio de Janeiro"},
}
CNPJ_BOM_PRECO = "11222333000181"
CNPJ_ESTRELA = "44555666000162"


def nota_item(nota_id: str, ts: datetime, gtin: str, preco: float, qtd: float, cnpj: str = CNPJ_BOM_PRECO) -> dict:
    """Monta uma linha de item de NFe a partir do catálogo — evita repetir unidade/município à mão."""
    produto = PRODUTOS[gtin]
    lojista = LOJISTAS[cnpj]
    return {
        "id": nota_id,
        "timestamp": ts.isoformat(),
        "gtin": gtin,
        "cnpj_lojista": cnpj,
        "preco_unitario": preco,
        "unidade_medida": produto["unidade"],
        "quantidade_comprada": qtd,
        "id_municipio": lojista["id_municipio"],
        "nome_municipio": lojista["nome_municipio"],
    }


def emitir_lote_csv(itens: list[dict], pasta: str = CAMINHO_LANDING) -> None:
    """Publica um lote de itens de NFe como um novo arquivo CSV (com header) na pasta 'landing'."""
    nome_arquivo = f"lote-{os.urandom(4).hex()}.csv"
    caminho_tmp = f"{pasta}/.{nome_arquivo}.tmp"
    caminho_final = f"{pasta}/{nome_arquivo}"
    with open(caminho_tmp, "w", newline="") as f:
        escritor = csv.DictWriter(f, fieldnames=COLUNAS_NFE)
        escritor.writeheader()
        for item in itens:
            escritor.writerow(item)
    os.rename(caminho_tmp, caminho_final)  # rename atômico: só agora o arquivo "existe" para o Spark
    print(f"📨 lote de {len(itens)} item(ns) publicado em landing_nfe/{nome_arquivo}")


schema_nfe_csv = StructType([
    StructField("id", StringType()),
    StructField("timestamp", TimestampType()),
    StructField("gtin", StringType()),
    StructField("cnpj_lojista", StringType()),
    StructField("preco_unitario", DoubleType()),
    StructField("unidade_medida", StringType()),
    StructField("quantidade_comprada", DoubleType()),
    StructField("id_municipio", StringType()),
    StructField("nome_municipio", StringType()),
])

# Data-base fixa e sintética: controlamos o event_time de cada item por completo,
# então o resultado é 100% determinístico — independe de QUANDO você rodar este notebook.
BASE_TIME = datetime(2026, 1, 1, 9, 0, 0)
rng = np.random.default_rng(42)  # mesma prática do resto do laboratório: NumPy com semente fixa

## Ruído de fundo: simulando "muitos dados" de outros produtos e lojistas

Um pipeline de verdade não processa só o caso que você está depurando — ele processa **tudo ao mesmo tempo**. Vamos gerar um volume razoável de vendas **aleatórias, mas reproduzíveis** (semente fixa do NumPy) para as outras combinações de produto/lojista, só para a tabela Gold nascer com bastante gente dentro.

⚠️ De propósito, esse gerador **evita** as duas combinações que vamos controlar à mão (Arroz/Bom Preço e Feijão/Bom Preço) — não queremos ruído aleatório contaminando os números que vamos conferir.

🧠 **Uma pegadinha de watermark que vale registrar:** o ruído deste lote tem que ter event time **anterior** ao que vamos processar em seguida (Exemplo 1). O watermark nunca recua — se publicássemos aqui um lote com timestamps *no futuro* (depois das 09:00), o watermark saltaria na frente, e os eventos do Exemplo 1 (09:00-09:45) chegariam **atrasados demais**, sendo tratados como dado tardio assim que os enxergássemos. Isso não é um capricho deste notebook: é a mesma regra do watermark que provamos no notebook 12, agora mordendo de verdade se você não respeitar a ordem cronológica ao simular.

In [ ]:
COMBINACOES_RUIDO = [
    (gtin, cnpj)
    for gtin in PRODUTOS
    for cnpj in LOJISTAS
    if not (cnpj == CNPJ_BOM_PRECO and gtin in (GTIN_ARROZ, GTIN_FEIJAO))
]


def gerar_ruido(n: int, inicio: datetime, fim: datetime) -> list[dict]:
    """Gera n itens de NFe aleatórios (produto/lojista/horário), evitando as combinações controladas."""
    duracao_segundos = int((fim - inicio).total_seconds())
    itens = []
    for i in range(n):
        gtin, cnpj = COMBINACOES_RUIDO[rng.integers(0, len(COMBINACOES_RUIDO))]
        produto = PRODUTOS[gtin]
        ts = inicio + timedelta(seconds=int(rng.uniform(0, duracao_segundos)))
        preco = round(float(rng.normal(produto["preco_base"], produto["preco_base"] * 0.03)), 2)
        qtd = float(rng.choice([1, 1, 1, 2, 2, 3, 4, 5]))
        itens.append(nota_item(f"NF-RUIDO-{i:04d}", ts, gtin, preco, qtd, cnpj))
    return itens


ruido_fase1 = gerar_ruido(24, BASE_TIME - timedelta(hours=1), BASE_TIME - timedelta(minutes=5))
emitir_lote_csv(ruido_fase1)

## O pipeline: join stream-static + quarentena (como antes) + acumulador com decaimento (novo)

Os dois primeiros estágios não mudam em relação à versão anterior deste notebook: enriquecer o stream com uma tabela de referência de preços e separar itens **válidos** de **suspeitos** com um filtro de faixa (0,5x–2,0x da referência). É o terceiro estágio que muda por completo.

### A fórmula: média ponderada recursiva

Em vez de agregar tudo de novo a cada lote, o acumulador guarda só dois números por chave — `valor_acumulado` e `quantidade_acumulada` — e cada novo lote os atualiza assim:

```
valor_acumulado_novo      = valor_acumulado_antigo × FATOR_DECAIMENTO + valor_do_lote
quantidade_acumulada_novo = quantidade_acumulada_antigo × FATOR_DECAIMENTO + quantidade_do_lote
preco_medio_ponderado     = valor_acumulado_novo / quantidade_acumulada_novo
```

Com `FATOR_DECAIMENTO = 0.8`, cada atualização "esquece" 20% do peso acumulado até então antes de somar o lote novo — é uma média móvel exponencialmente ponderada (o mesmo princípio por trás de um EWMA), só que expressa em cima de `valor × quantidade` em vez de um preço já pronto. Dois efeitos surgem dessa única fórmula:

- **Curto prazo:** um lote isolado (mesmo um grande, tipo uma venda por atacado) nunca vira o preço sozinho — ele só desloca um acumulado que já carrega a história recente.
- **Longo prazo:** como o peso antigo decai geometricamente, uma mudança de preço **genuína e sustentada** eventualmente domina o acumulado — a "memória" do pipeline tem um horizonte, não é infinita.

⚠️ **Por que não usar uma média cumulativa sem decaimento (`FATOR_DECAIMENTO = 1.0`)?** Ela também resiste a outliers de curto prazo — mas resiste *demais* a mudanças reais: se o lojista sobe o preço de vez no meio do dia, uma média que nunca esquece o passado demora muito mais para acompanhar. Vamos provar isso com números no Exemplo 2.

In [ ]:
import pandas as pd
from pyspark.sql import Row
from pyspark.sql.functions import avg, col, count, round as spark_round, sum as spark_sum
from pyspark.sql.streaming.state import GroupStateTimeout

FATOR_DECAIMENTO = 0.8
JANELA_INATIVIDADE = "1 hour"  # mesmo horizonte do watermark: sem vendas por 1h, o estado se encerra

# Tabela de referência: preço de catálogo de cada produto — um DataFrame comum, não streaming
tabela_preco_referencia = spark.createDataFrame([
    Row(gtin=gtin, nome_produto=info["nome"], preco_referencia=info["preco_base"])
    for gtin, info in PRODUTOS.items()
])

itens_stream = (
    spark.readStream
    .format("csv")
    .option("header", True)
    .option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ss")
    .schema(schema_nfe_csv)
    .load(CAMINHO_LANDING)
    .withWatermark("timestamp", JANELA_INATIVIDADE)   # 1 hora — tolera notas atrasadas, não os 15min de antes
)

# Estágio 1: enriquecer — join stream-static por gtin (idêntico à versão anterior)
itens_enriquecidos = itens_stream.join(tabela_preco_referencia, on="gtin", how="left")

# Estágio 2: separar — dentro de 50%-200% do preço de referência é "válido"; fora disso é "suspeito"
dentro_da_faixa = (
    (col("preco_unitario") >= col("preco_referencia") * 0.5)
    & (col("preco_unitario") <= col("preco_referencia") * 2.0)
)
itens_validos = itens_enriquecidos.filter(dentro_da_faixa)
itens_suspeitos = itens_enriquecidos.filter(~dentro_da_faixa)

### O acumulador com estado (`applyInPandasWithState`)

A função abaixo é chamada **uma vez por chave, por micro-batch** (só para chaves que receberam itens novos), recebendo o estado da chamada anterior e devolvendo o estado atualizado — exatamente a fórmula recursiva de cima. Ela também trata o caso de **timeout**: se `gtin`+`cnpj_lojista` ficar 1h sem vender nada, o Spark chama a função de novo com `state.hasTimedOut = True` e nenhum dado novo — é nossa deixa para emitir um registro final e liberar a memória daquele estado.

In [ ]:
schema_estado_acumulado = StructType([
    StructField("valor_acumulado", DoubleType()),
    StructField("quantidade_acumulada", DoubleType()),
    StructField("ultima_atualizacao", TimestampType()),
])

schema_saida_acumulada = StructType([
    StructField("gtin", StringType()),
    StructField("cnpj_lojista", StringType()),
    StructField("preco_medio_ponderado", DoubleType()),
    StructField("quantidade_acumulada", DoubleType()),
    StructField("qtd_itens_no_lote", DoubleType()),
    StructField("atualizado_em", TimestampType()),
    StructField("motivo", StringType()),
])


def atualizar_preco_medio(key, pdf_iter, state):
    gtin, cnpj_lojista = key

    if state.hasTimedOut:
        # Mais de 1h sem vendas para esta chave — emite um registro final e libera o estado.
        if state.exists:
            valor_acc, qtd_acc, ultima = state.get
            yield pd.DataFrame([{
                "gtin": gtin,
                "cnpj_lojista": cnpj_lojista,
                "preco_medio_ponderado": valor_acc / qtd_acc if qtd_acc else None,
                "quantidade_acumulada": qtd_acc,
                "qtd_itens_no_lote": 0.0,
                "atualizado_em": ultima,
                "motivo": "expirado_por_inatividade",
            }])
            state.remove()
        return

    pdf = pd.concat(pdf_iter)
    valor_lote = float((pdf["preco_unitario"] * pdf["quantidade_comprada"]).sum())
    qtd_lote = float(pdf["quantidade_comprada"].sum())
    ultima_ts = pdf["timestamp"].max()

    if state.exists:
        valor_ant, qtd_ant, _ = state.get
        valor_acc = valor_ant * FATOR_DECAIMENTO + valor_lote
        qtd_acc = qtd_ant * FATOR_DECAIMENTO + qtd_lote
    else:
        valor_acc, qtd_acc = valor_lote, qtd_lote

    state.update((valor_acc, qtd_acc, ultima_ts))
    # Reagenda o timeout para 1h a partir do event time DESTA venda — não do watermark atual
    # (o watermark já está sempre ~1h atrás do relógio, então "watermark + 1h" seria quase "agora").
    state.setTimeoutTimestamp(int(ultima_ts.timestamp() * 1000) + 60 * 60 * 1000)

    yield pd.DataFrame([{
        "gtin": gtin,
        "cnpj_lojista": cnpj_lojista,
        "preco_medio_ponderado": valor_acc / qtd_acc,
        "quantidade_acumulada": qtd_acc,
        "qtd_itens_no_lote": float(len(pdf)),
        "atualizado_em": ultima_ts,
        "motivo": "atualizacao",
    }])


resultado_acumulado = (
    itens_validos
    .groupBy("gtin", "cnpj_lojista")
    .applyInPandasWithState(
        atualizar_preco_medio,
        outputStructType=schema_saida_acumulada,
        stateStructType=schema_estado_acumulado,
        outputMode="Update",
        timeoutConf=GroupStateTimeout.EventTimeTimeout,
    )
)

### Por que `foreachBatch` em vez de `format("parquet")` direto

`applyInPandasWithState` com `outputMode="Update"` só pode alimentar sinks que aceitam atualizações — e o sink de arquivo (`parquet`, `csv`, `json`) só aceita `append`. A solução não é escrever menos, é escrever **diferente**: com `foreachBatch`, cada micro-batch vira um DataFrame comum de batch, que gravamos com `.write.mode("append")`. A tabela Gold passa a ser um **log histórico** — cada atualização de cada chave vira uma linha nova, nunca sobrescrita. Para saber "o preço atual", basta pegar a linha mais recente por chave (fazemos isso mais adiante).

In [ ]:
def gravar_lote_no_gold(lote_df, id_do_lote):
    if lote_df.isEmpty():
        return
    lote_df.write.mode("append").parquet(CAMINHO_GOLD)


query_gold = (
    resultado_acumulado.writeStream
    .foreachBatch(gravar_lote_no_gold)
    .outputMode("update")
    .option("checkpointLocation", CAMINHO_CHECKPOINT_GOLD)
    .trigger(processingTime="2 seconds")
    .start()
)

# Quarentena: os itens suspeitos, para auditoria — sink memory (é só para inspecionarmos aqui no notebook)
query_quarentena = (
    itens_suspeitos
    .select(
        "id", "timestamp", "nome_produto", "cnpj_lojista",
        "preco_unitario", "preco_referencia", "quantidade_comprada",
    )
    .withColumn("razao_sobre_referencia", spark_round(col("preco_unitario") / col("preco_referencia"), 3))
    .writeStream
    .format("memory")
    .queryName("quarentena")
    .outputMode("append")
    .trigger(processingTime="2 seconds")
    .start()
)

### Ritmo dos lotes: esperar o micro-batch, não adivinhar um `sleep`

Nos notebooks 12 e 13, um `time.sleep(5)` fixo entre eventos bastava. Aqui isso não é confiável: `applyInPandasWithState` + `foreachBatch` têm um custo de processamento por lote maior (serialização Arrow, worker Python, escrita em disco), e se dois lotes forem publicados rápido demais, o file source os lê **juntos** no mesmo micro-batch — e viram uma única atualização de estado, não duas. Como cada exemplo abaixo depende de ver o acumulado evoluir **lote a lote**, precisamos ter certeza de que cada arquivo foi processado sozinho antes de publicar o próximo.

A solução: em vez de dormir um tempo fixo, **esperar o progresso real da query** — `query.lastProgress` nos diz o `batchId` e quantas linhas de entrada o último micro-batch teve. Publicamos um lote e esperamos até ver um `batchId` novo com `numInputRows > 0`.

In [ ]:
def aguardar_processamento(query, timeout: int = 40) -> None:
    """Espera a query processar um novo micro-batch com dados novos, em vez de adivinhar um sleep fixo."""
    batch_visto = query.lastProgress["batchId"] if query.lastProgress else -1
    inicio = time.time()
    while time.time() - inicio < timeout:
        time.sleep(1)
        progresso = query.lastProgress
        if progresso and progresso["batchId"] > batch_visto and progresso.get("numInputRows", 0) > 0:
            time.sleep(1)  # pequena folga para o foreachBatch concluir a escrita em disco
            return
    print(f"⚠️  tempo esgotado ({timeout}s) esperando o próximo micro-batch — a consulta seguinte pode vir incompleta")


# Drena o ruído de fundo publicado lá no início, antes de começar os cenários controlados
aguardar_processamento(query_gold)

## Exemplo 1: robustez de curto prazo — atacado e aberração, no mesmo produto

Cenário no Mercado Bom Preço, entre 09:00 e 09:45: 3 vendas de varejo normais (~R\$ 26,50), depois uma compra por atacado (40 unidades a R\$ 23,50 — desconto de volume legítimo), depois um **erro de digitação** (R\$ 2.650,00 em vez de R\$ 26,50 — dois dígitos a mais), e mais 4 vendas de varejo normais para fechar. Cada linha é um lote separado — simulando 5 em 5 minutos.

In [ ]:
ticks_exemplo1 = [
    ("NF-3001", 2, 26.50, 1.0, "T1 — varejo normal"),
    ("NF-3002", 7, 26.60, 1.0, "T2 — varejo normal"),
    ("NF-3003", 12, 26.40, 1.0, "T3 — varejo normal"),
    ("NF-3004", 20, 23.50, 40.0, "T4 — compra por atacado (40 unidades, desconto de volume)"),
    ("NF-3005", 25, 2650.00, 1.0, "T5 — ABERRAÇÃO: dois dígitos a mais no caixa"),
    ("NF-3006", 30, 26.70, 1.0, "T6 — varejo normal"),
    ("NF-3007", 35, 26.35, 1.0, "T7 — varejo normal"),
    ("NF-3008", 40, 26.55, 1.0, "T8 — varejo normal"),
    ("NF-3009", 45, 26.45, 1.0, "T9 — varejo normal"),
]

for nota_id, minuto, preco, qtd, rotulo in ticks_exemplo1:
    print(f"--- {rotulo} ---")
    emitir_lote_csv([nota_item(nota_id, BASE_TIME + timedelta(minutes=minuto), GTIN_ARROZ, preco, qtd)])
    aguardar_processamento(query_gold)

In [ ]:
spark.table("quarentena") \
    .select("id", "preco_unitario", "preco_referencia", "razao_sobre_referencia") \
    .orderBy("id") \
    .show(truncate=False)

historico_arroz_bompreco = (
    spark.read.parquet(CAMINHO_GOLD)
    .filter((col("gtin") == GTIN_ARROZ) & (col("cnpj_lojista") == CNPJ_BOM_PRECO))
    .select("atualizado_em", "qtd_itens_no_lote", spark_round("preco_medio_ponderado", 4).alias("preco_medio_ponderado"), spark_round("quantidade_acumulada", 4).alias("quantidade_acumulada"))
    .orderBy("atualizado_em")
)
historico_arroz_bompreco.show(truncate=False)

📌 **A aberração (`NF-3005`) está na quarentena — e só ela.** As 8 vendas legítimas nunca aparecem lá; estão do outro lado do filtro, alimentando o acumulador.

📌 **A trajetória do acumulado conta a história:** depois de T1-T3 o preço médio está estável em ~R\$ 26,50. O atacado (T4) puxa para ~R\$ 23,64 — um deslocamento real, porque 40 unidades pesam de verdade. A aberração (T5) **nem aparece** na tabela — ela nunca chega a atualizar o estado, porque foi barrada na quarentena antes. Nos lotes seguintes (T6-T9), o preço médio sobe devagar, de volta em direção a R\$ 26,50 — o efeito do atacado vai perdendo peso a cada atualização (decaimento de 20%), mas não desaparece de uma vez: é exatamente esse equilíbrio entre "reagir rápido" e "não esquecer tudo de uma vez" que o fator de decaimento controla.

## Preenchendo a lacuna: mais ruído entre os dois cenários

O Arroz no Mercado Bom Preço parou de vender às 09:45. Antes de pular para o Exemplo 2 (que só começa às 11:00), publicamos mais um lote de ruído de fundo cobrindo esse intervalo — mantendo o event time sempre avançando, e de quebra empurrando o watermark adiante o suficiente para o `GroupStateTimeout` do Arroz ter uma chance real de disparar (conferimos isso mais adiante).

In [ ]:
ruido_fase2 = gerar_ruido(20, BASE_TIME + timedelta(minutes=50), BASE_TIME + timedelta(hours=1, minutes=55))
emitir_lote_csv(ruido_fase2)
aguardar_processamento(query_gold)

## Exemplo 2: adaptação de longo prazo — quando o preço muda de verdade

Agora um cenário diferente: o Feijão no Mercado Bom Preço vende a ~R\$ 7,50 por um tempo, depois o lojista **genuinamente** sobe o preço para a faixa de R\$ 9,00 — e nunca mais volta. Isso não é ruído nem erro: é uma mudança real de prática que o acumulado precisa **acompanhar**, mesmo carregando toda a história anterior.

Vamos simular 16 lotes de 5 em 5 minutos (4 no preço antigo, 12 no preço novo) e comparar, lado a lado, o que o **decaimento de 0,8** produz contra o que uma **média cumulativa sem decaimento** (que nunca esquece nada) produziria com os mesmíssimos dados.

In [ ]:
INICIO_FEIJAO = BASE_TIME + timedelta(hours=2)  # 11:00 — simulando "mais tarde no mesmo dia"

# T1-T4: preço antigo, ~R$ 7,50
precos_antigos = [7.50, 7.45, 7.55, 7.50]
for i, preco in enumerate(precos_antigos):
    minuto = i * 5
    print(f"--- T{i + 1} (preço antigo) ---")
    emitir_lote_csv([nota_item(f"NF-40{i:02d}1", INICIO_FEIJAO + timedelta(minutes=minuto), GTIN_FEIJAO, preco, 2.0)])
    aguardar_processamento(query_gold)

A partir daqui, o lojista sobe o preço de vez — e não volta mais ao valor antigo.

In [ ]:
# T5-T16: preço novo e permanente, ~R$ 9,00
precos_novos = [9.00, 9.05, 8.95, 9.00, 9.10, 8.90, 9.00, 9.05, 8.95, 9.00, 9.05, 8.95]
for i, preco in enumerate(precos_novos):
    minuto = 20 + i * 5
    print(f"--- T{i + 5} (preço novo) ---")
    emitir_lote_csv([nota_item(f"NF-40{i:02d}5", INICIO_FEIJAO + timedelta(minutes=minuto), GTIN_FEIJAO, preco, 2.0)])
    aguardar_processamento(query_gold)

In [ ]:
historico_feijao_bompreco = (
    spark.read.parquet(CAMINHO_GOLD)
    .filter((col("gtin") == GTIN_FEIJAO) & (col("cnpj_lojista") == CNPJ_BOM_PRECO))
    .select("atualizado_em", spark_round("preco_medio_ponderado", 4).alias("preco_medio_ponderado_COM_decaimento"))
    .orderBy("atualizado_em")
)
historico_feijao_bompreco.show(20, truncate=False)

### E se não houvesse decaimento?

A tabela acima é o resultado **real** do pipeline (decaimento 0,8). Para comparar, eis a mesma sequência de 16 lotes recalculada em Python puro com `FATOR_DECAIMENTO = 1.0` (uma média cumulativa que nunca esquece) — não é Spark, é só a mesma fórmula recursiva, aplicada aos mesmos números, para conferência lado a lado.

In [ ]:
def simular_acumulador(precos_e_quantidades, fator_decaimento):
    valor_acc, qtd_acc, existe = 0.0, 0.0, False
    trajetoria = []
    for preco, qtd in precos_e_quantidades:
        valor_lote, qtd_lote = preco * qtd, qtd
        if existe:
            valor_acc = valor_acc * fator_decaimento + valor_lote
            qtd_acc = qtd_acc * fator_decaimento + qtd_lote
        else:
            valor_acc, qtd_acc, existe = valor_lote, qtd_lote, True
        trajetoria.append(round(valor_acc / qtd_acc, 4))
    return trajetoria


eventos_feijao = [(7.50, 2.0), (7.45, 2.0), (7.55, 2.0), (7.50, 2.0)] + [(p, 2.0) for p in precos_novos]

comparativo = pd.DataFrame({
    "lote": [f"T{i+1}" for i in range(len(eventos_feijao))],
    "preco_do_lote": [p for p, _ in eventos_feijao],
    "preco_medio_COM_decaimento_08": simular_acumulador(eventos_feijao, 0.8),
    "preco_medio_SEM_decaimento_10": simular_acumulador(eventos_feijao, 1.0),
})
spark.createDataFrame(comparativo).show(20, truncate=False)

📌 **O contraste que prova o ponto:** depois dos mesmos 12 lotes a ~R\$ 9,00, o acumulado **com decaimento** termina em **R\$ 8,93** — já bem próximo da nova realidade. O acumulado **sem decaimento** termina em **R\$ 8,63** — ainda visivelmente puxado para baixo pelos 4 lotes antigos a R\$ 7,50, porque uma média que nunca esquece dá o mesmo peso a uma venda de agora e a uma venda de horas atrás, para sempre.

🧠 **O fator de decaimento é um dial, não uma constante mágica.** Mais perto de 1,0 → mais memória, mais suavização, mais lento para acompanhar mudanças reais. Mais perto de 0,0 → esquece rápido, acompanha mudanças quase imediatamente, mas fica mais sensível a ruído de curto prazo (o mesmo problema que o Exemplo 1 mostrou). `0,8` é um meio-termo razoável para um produto que vende algumas vezes por hora — o valor certo depende de quão rápido o negócio realmente muda de preço.

## Lendo "o preço atual": a linha mais recente por chave

Como a tabela Gold é um log (cada atualização é uma linha nova, nunca sobrescrita), "o preço atual" de um produto/lojista é a linha com o `atualizado_em` mais recente **daquela chave** — um `row_number()` particionado por `gtin`+`cnpj_lojista` resolve isso, o mesmo padrão que você usaria para achar a versão mais recente de qualquer registro num log append-only (Kafka compactado, cada Delta/Iceberg por baixo dos panos, etc.).

In [ ]:
from pyspark.sql import Window
from pyspark.sql.functions import desc, row_number

gold_completo = spark.read.parquet(CAMINHO_GOLD)
print(f"Total de linhas no histórico Gold: {gold_completo.count()}")

mais_recente_por_chave = Window.partitionBy("gtin", "cnpj_lojista").orderBy(desc("atualizado_em"))

preco_atual = (
    gold_completo
    .filter(col("motivo") == "atualizacao")
    .withColumn("rn", row_number().over(mais_recente_por_chave))
    .filter(col("rn") == 1)
    .select(
        "gtin", "cnpj_lojista",
        spark_round("preco_medio_ponderado", 2).alias("preco_atual"),
        spark_round("quantidade_acumulada", 2).alias("peso_acumulado"),
        "atualizado_em",
    )
    .orderBy("gtin", "cnpj_lojista")
)
preco_atual.show(truncate=False)

📌 Essa é a tabela que um painel de preços leria: uma linha por produto/lojista, sempre a mais recente — mesmo que o arquivo físico por baixo seja um log que só cresce. É o mesmo truque que sustenta compactação de tópicos Kafka e a camada de metadados do Delta Lake/Iceberg: nunca é preciso reescrever o passado, só saber ler "a última versão".

## Conferindo o `GroupStateTimeout`: chaves que ficam quietas se encerram sozinhas

Nosso Exemplo 1 parou de vender Arroz no Mercado Bom Preço às 09:45. Como o resto da simulação avança o relógio (event time) bem além de 10:45 — mais de 1h depois — essa chave deveria ter recebido um registro de encerramento por inatividade, com `motivo = "expirado_por_inatividade"`, e seu estado deveria ter sido liberado.

In [ ]:
spark.read.parquet(CAMINHO_GOLD) \
    .filter(col("motivo") == "expirado_por_inatividade") \
    .select("gtin", "cnpj_lojista", spark_round("preco_medio_ponderado", 2).alias("preco_medio_ponderado"), "atualizado_em") \
    .show(truncate=False)

📌 Se a linha do Arroz/Bom Preço aparece aqui, é a prova de que o `GroupStateTimeout.EventTimeTimeout` funcionou de ponta a ponta: o watermark avançou além de `ultima_atualizacao + 1h` daquela chave, o Spark chamou nossa função com `state.hasTimedOut = True`, e nós emitimos o registro final e liberamos a memória — sem isso, um pipeline rodando por meses acumularia estado para todo produto/lojista que já vendeu **uma vez**, para sempre.

## Recapitulando: o pipeline completo

| Estágio | Técnica | Por quê |
|---|---|---|
| Enriquecer | `join` stream-static | Trazer o preço de referência sem precisar embuti-lo em cada evento |
| Separar | `filter` por faixa (0,5x-2,0x da referência) | Pegar erro de digitação/glitch **antes** de qualquer conta |
| Auditar | sink `memory` para os suspeitos | Dado ruim não devia ser só descartado — devia poder ser revisado |
| Acumular | `applyInPandasWithState` com decaimento exponencial | Média ponderada que carrega a história do dia, sem deixar um lote isolado dominar |
| Encerrar | `GroupStateTimeout.EventTimeTimeout` (1h, = watermark) | Liberar memória de chaves que pararam de vender |
| Persistir | `foreachBatch` + `format("parquet")` em `append` | Sink de arquivo só aceita append — tratamos a tabela como log histórico |
| Consultar | `row_number()` particionado por chave, ordenado por `atualizado_em` | "Preço atual" = linha mais recente por chave, sem reescrever nada |

🧠 **A lição central:** "não sofrer efeito de outliers" e "refletir a prática ao longo do tempo" são objetivos em tensão, não a mesma coisa — resolver os dois ao mesmo tempo exige memória (para suavizar) *e* esquecimento (para não ficar preso ao passado). O decaimento exponencial é o parâmetro que governa esse equilíbrio.

In [ ]:
query_gold.stop()
query_quarentena.stop()
# Encerra a SparkSession — libera threads e memória
spark.stop()

---
🎉 **Acumulador de preço médio com estado concluído!** Você praticou:

- **Watermark de 1 hora** para tolerar notas atrasadas, bem mais generoso que o ritmo de 5 minutos dos lotes
- **`applyInPandasWithState`** — estado arbitrário por chave, atualizado lote a lote, sem reprocessar o histórico inteiro
- **Média ponderada recursiva com decaimento exponencial** — e por que ela resolve outliers de curto prazo *e* mudanças reais de longo prazo, ao mesmo tempo
- **`GroupStateTimeout.EventTimeTimeout`** — encerrar e liberar estado de chaves inativas, usando o mesmo horizonte do watermark
- **`foreachBatch` como ponte** entre um sink que só aceita `append` e uma agregação que naturalmente "atualiza"
- **Padrão de leitura "última versão por chave"** — o mesmo truque por trás de tópicos Kafka compactados e das camadas de metadados do Delta Lake/Iceberg

📌 Todo o estado desta simulação vive em `../data/streaming/nb14` e `../data/gold/nb14_preco_medio_produto_lojista_hora` (ambos git-ignorados) e é limpo automaticamente toda vez que este notebook roda do início.